# 16. Reproducible scientific evaluators and numerical contracts

![Reproducible evaluator](../images/16_reproducible_scientific_evaluators.svg)

This notebook validates a synthetic 28-row iso-catalog registry and a matched continuous retrieval score. Read the [lecture](../lectures/16_reproducible_scientific_evaluators.md) and return to the [tutorial index](../README.md).

**Learning goals:** create a canonical digest, validate the 28-model allocation registry, reject provenance mismatches, and verify that GFC and independent completion use a comparable gallery margin.

In [ ]:
import hashlib
import json
import numpy as np

SEED = 31
rng = np.random.default_rng(SEED)
assert rng.integers(0, 2) in {0, 1}


## Canonical bytes

Digests establish exact identity. They do not establish that the experimental design is scientifically sensible, so both levels need validation.

In [ ]:
def digest(value):
    payload = json.dumps(value, sort_keys=True, separators=(',', ':')).encode()
    return hashlib.sha256(payload).hexdigest()

assert digest({'b': 2, 'a': 1}) == digest({'a': 1, 'b': 2})


## The 28-row registry

Eight blocks contain breadth, balanced, and phase depth. Four prespecified blocks contain nearby jitter. Each row records the meaning of its data, not just a filename.

In [ ]:
ALLOCATIONS = {
    'breadth': (250_000, 1, 'base_phase'),
    'balanced': (125_000, 2, 'phase_separated'),
    'phase_depth': (62_500, 4, 'phase_separated'),
    'nearby_jitter': (62_500, 4, 'nearby_jitter'),
}

def make_row(block, allocation):
    unique_sequences, origins_per_sequence, origin_policy = ALLOCATIONS[allocation]
    return {
        'block': block, 'allocation': allocation,
        'unique_sequences': unique_sequences, 'origins_per_sequence': origins_per_sequence,
        'nominal_catalog_size': unique_sequences * origins_per_sequence,
        'origin_policy': origin_policy, 'planned_exposure': 4_096_000,
        'optimization_seed': 100 + block, 'replicate_seed': 200 + block,
        'train_manifest_digest': digest({'block': block, 'allocation': allocation}),
        'phase_catalog_digest': 'phase-v1',
        'sequence_stream_version': 'sequence-v2', 'phase_stream_version': 'phase-v1',
        'spatial_stream_version': 'spatial-v1', 'mask_stream_version': 'mask-v1',
    }

registry = [make_row(block, allocation) for block in range(8) for allocation in ('breadth', 'balanced', 'phase_depth')]
registry += [make_row(block, 'nearby_jitter') for block in range(4)]
assert len(registry) == 28


## Cross-row checks

Each field can be valid while the full registry is wrong. The validator checks the expected cells, paired seeds, fixed exposure, and phase-depth versus jitter matching.

In [ ]:
def validate_registry(rows):
    expected = {(block, allocation) for block in range(8) for allocation in ('breadth', 'balanced', 'phase_depth')} | {(block, 'nearby_jitter') for block in range(4)}
    observed = {(row['block'], row['allocation']) for row in rows}
    if observed != expected or len(rows) != len(expected):
        raise ValueError('registry does not contain the frozen 28 rows')
    for row in rows:
        if row['nominal_catalog_size'] != row['unique_sequences'] * row['origins_per_sequence']:
            raise ValueError('inconsistent nominal catalog')
        if row['nominal_catalog_size'] != 250_000:
            raise ValueError('all rows must have the same nominal catalog')
    for block in range(4):
        phase = next(row for row in rows if row['block'] == block and row['allocation'] == 'phase_depth')
        jitter = next(row for row in rows if row['block'] == block and row['allocation'] == 'nearby_jitter')
        for key in ('unique_sequences', 'planned_exposure', 'optimization_seed', 'replicate_seed', 'phase_catalog_digest'):
            if phase[key] != jitter[key]:
                raise ValueError('phase and jitter must be paired')

validate_registry(registry)
try:
    broken = [dict(row) for row in registry]; broken[-1]['planned_exposure'] = 8_192_000
    validate_registry(broken)
except ValueError:
    pass
else:
    raise AssertionError('mismatched jitter exposure must fail')


## Matched continuous gallery margin

Both GFC and independent completion need the same target-versus-competitor score. A rank alone discards useful information about confidence.

In [ ]:
def target_margin(distances, target_index):
    distances = np.asarray(distances, dtype=float)
    target = distances[target_index]
    competitor = np.min(np.delete(distances, target_index))
    return float(competitor - target)

gfc_margin = target_margin([0.7, 0.4, 0.9, 1.1], 1)
completion_margin = target_margin([0.8, 0.5, 0.9, 1.2], 1)
assert gfc_margin > 0 and completion_margin > 0
assert np.isclose(gfc_margin - completion_margin, 0.0)


**Takeaway:** reproducibility requires a registry that names the scientific intervention, cross-row validation that protects paired comparisons, and an evaluator whose GFC and control scores share a defined scale.

Previous: [15. Exposure and replication](15_exposure_and_replication.ipynb) · Next: [17. Iso-catalog allocation](17_hierarchical_support_and_factorial_inference.ipynb)